# Risk-Aware MARL — Perception Module Ablation
[![Paper](https://img.shields.io/badge/MDPI-Mathematics-blue)](https://www.mdpi.com/journal/mathematics)

Trains and evaluates the four encoder variants from **Table 5** of the paper:

|Variant             |     F1    |  Acc    |  Prec  |  Rec  | Description    |
 | ------------------|-----------|---------|--------|-------|----------------|
 | `radar_cnn`       |   0.7728  | 0.7896  |0.7667  | 0.7836|Radar-only CNN baseline |
 | `multimodal_cnn`  |   0.8403  | 0.8451  |0.8440  | 0.8302|Radar + satellite CNN fusion|
 | `it_single`       |   0.8469  | 0.8596  |0.8457  | 0.8388| Single-stream ViT (radar only) |
 | `vit_multimodal`  |   0.8800  | 0.8898  |0.8807  | 0.8701| Two-stream ViT — **proposed** |


## 0 · Setup

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time

import numpy as np

if shutil.which("nvidia-smi") is not None:
    r = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True,
        text=True,
    )
    if r.returncode == 0 and r.stdout.strip():
        print("GPU:", r.stdout.strip())
    else:
        print("GPU info unavailable. Continuing with CPU if needed.")
else:
    print("No GPU detected. ViT training is slower on CPU.")
    print("Use a GPU runtime in Colab for faster results.")

print("Python:", sys.version.split()[0])

In [ ]:
# ── GitHub Setup — runs automatically on Colab; safe to skip locally ──────────
import os, subprocess, sys

REPO_URL = "https://github.com/aliakarma/agentic-weather-rl.git"
REPO_DIR = "agentic-weather-rl"

def _run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stderr.strip())
    return result

# ── 1. Determine REPO_ROOT robustly (idempotent — safe to re-run) ─────────────
# Walk up the directory tree to find the repo root, so re-running this cell
# never nests deeper into the directory structure.
_here = os.getcwd()
_found = False
_check = _here
for _ in range(6):                            # search up to 6 levels up
    if os.path.basename(_check) == REPO_DIR and os.path.isdir(os.path.join(_check, "src")):
        REPO_ROOT = _check
        _found = True
        break
    _check = os.path.dirname(_check)

if not _found:
    # Not inside the repo yet — clone if needed, then enter
    if not os.path.exists(os.path.join(_here, REPO_DIR)):
        print(f"Cloning {REPO_URL} ...")
        r = _run(f"git clone --depth 1 {REPO_URL} {REPO_DIR}")
        print("✓ Clone complete" if r.returncode == 0 else "✗ Clone failed")
    else:
        print(f"✓ Repo already present at ./{REPO_DIR}")
    REPO_ROOT = os.path.join(_here, REPO_DIR)

os.chdir(REPO_ROOT)
print(f"  Working directory: {os.getcwd()}")

# ── 2. Remove torch.py mock (CRITICAL — shadows real PyTorch) ─────────────────
for _f in [os.path.join(REPO_ROOT, "torch.py"),
           os.path.join(REPO_ROOT, "torch.pyc")]:
    if os.path.exists(_f):
        os.remove(_f)
        print(f"✓ Removed {os.path.basename(_f)} (was shadowing real PyTorch)")
_bad = [k for k in sys.modules if (k == "torch" or k.startswith("torch."))
        and not getattr(sys.modules[k], "__path__", None)]
for _k in _bad:
    del sys.modules[_k]
if _bad:
    print(f"✓ Evicted stale torch mock from sys.modules ({len(_bad)} entries)")

# ── 3. Install dependencies ───────────────────────────────────────────────────
print("Installing requirements...")
_run(f"{sys.executable} -m pip install -q -r requirements.txt")
print("✓ Requirements installed")

# ── 4. Add repo root to Python path ──────────────────────────────────────────
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print("✓ sys.path updated")

# ── 5. Device detection ───────────────────────────────────────────────────────
try:
    import torch as _torch
    DEVICE = "cuda" if _torch.cuda.is_available() else "cpu"
except Exception:
    DEVICE = "cpu"
print(f"✓ Device: {DEVICE}")
print("Repository ready. You can now run all cells below.")


In [ ]:
import os, subprocess

# 1. Download Logic
HF_SUBSET_URL  = "https://huggingface.co/datasets/aliakarma/SEVIR/resolve/main/sevir_subset.h5"
HF_SUBSET_DIR  = "data/huggingface_subset"
HF_SUBSET_FILE = os.path.join(HF_SUBSET_DIR, "sevir_subset.h5")

os.makedirs(HF_SUBSET_DIR, exist_ok=True)
if not os.path.exists(HF_SUBSET_FILE):
    print(f"Downloading subset to {HF_SUBSET_FILE}...")
    subprocess.run(["wget", "-O", HF_SUBSET_FILE, HF_SUBSET_URL], capture_output=True)
    print("✓ Download complete")
else:
    print("✓ Dataset already present")

# 2. Configuration Logic (Hugging Face Subset Only)
DATA_DIR   = HF_SUBSET_DIR
EPOCHS     = 50
BATCH      = 16
LR         = 1e-4
OUTPUT_DIM = 128

CHECKPOINT_DIR, RESULTS_DIR = "checkpoints", "results/example_results"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"Active Data Dir: {DATA_DIR}")
print(f"Epochs increased to: {EPOCHS}")
print(f"Device: {DEVICE if 'DEVICE' in dir() else 'cpu'}")

## 1 · Dataset Configuration

In [ ]:
if 'DEVICE' not in dir():
    DEVICE = 'cpu'

# ── Data paths ─────────────────────────────────────────────────────────────────
# 1. Full SEVIR (requires AWS CLI/S3 download)
# 2. Hugging Face Subset (sevir_subset.h5)
# 3. Sample/Synthetic (Fallback)

USE_FULL_SEVIR = False
USE_HF_SUBSET  = True   # Set to True for the downloaded HF data


if USE_HF_SUBSET:
    DATA_DIR = "data/huggingface_subset"
    EPOCHS   = 50
    BATCH    = 16


LR         = 1e-4
OUTPUT_DIM = 128
CHECKPOINT_DIR = "checkpoints"
RESULTS_DIR    = "results/example_results"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"Data dir   : {DATA_DIR}")
print(f"Epochs     : {EPOCHS}")
print(f"Batch size : {BATCH}")
print(f"Device     : {DEVICE}")
print()

if USE_HF_SUBSET:
    print("NOTE: Using Hugging Face SEVIR subset.")
elif not USE_FULL_SEVIR:
    print("NOTE: Using sample dataset / synthetic fallback.")

## 2 · Train All Encoder Variants

In [ ]:
VARIANTS = ["radar_cnn", "multimodal_cnn", "vit_single", "vit_multimodal"]

EXPECTED_F1 = {
    "radar_cnn":      0.77,
    "multimodal_cnn": 0.84,
    "vit_single":     0.85,
    "vit_multimodal": 0.88,
}

variant_results   = {}
variant_histories = {}

for variant in VARIANTS:
    print(f"\n{chr(9472)*55}")
    print(f"  Variant: {variant}  (expected F1 \u2248 {EXPECTED_F1[variant]})")
    print(f"{chr(9472)*55}")

    ckpt = f"{CHECKPOINT_DIR}/perception_encoder_{variant}.npz"
    t0   = time.time()

    extra = ["--use_sample"] if not USE_FULL_SEVIR else []
    cmd   = [
        sys.executable, "-m", "src.models.vit_encoder",
        "--mode",       "train",
        "--data_dir",   DATA_DIR,
        "--checkpoint", ckpt,
        "--epochs",     str(EPOCHS),
        "--lr",         str(LR),
        "--batch_size", str(BATCH),
        "--output_dim", str(OUTPUT_DIM),
        "--model_type", variant,
    ] + extra

    # capture_output=True so Colab shows the training logs in the cell output
    result  = subprocess.run(cmd, capture_output=True, text=True, cwd=REPO_ROOT)
    elapsed = time.time() - t0

    print(result.stdout)          # show all training logs
    if result.stderr.strip():
        print(result.stderr[-500:])   # show any errors

    if result.returncode != 0:
        print(f"  ✘ Training failed for {variant}")
    else:
        print(f"  Training time: {elapsed/60:.1f} min")
    variant_results[variant] = {"checkpoint": ckpt, "elapsed": elapsed}

print("All variants trained.")

## 3 · Evaluate All Variants

In [ ]:
eval_metrics = {}

for variant in VARIANTS:
    # Construct checkpoint path *without* the .npz extension, as the script adds it.
    ckpt = f"{CHECKPOINT_DIR}/perception_encoder_{variant}"

    print(f"\nEvaluating {variant}...")

    result = subprocess.run(
        [sys.executable, "-m", "src.models.vit_encoder",
         "--mode",       "eval",
         "--data_dir",   DATA_DIR,
         "--checkpoint", ckpt, # Pass path without .npz
         "--split",      "test",
         "--model_type", variant]
        + (["--use_sample"] if not USE_FULL_SEVIR else []),
        capture_output=True, text=True,
        cwd=REPO_ROOT,      # always run from repo root regardless of current dir
    )
    print(result.stdout[-2000:])
    if result.stderr.strip():
        print("STDERR:", result.stderr[-500:])

    # Parse metrics from stdout
    f1 = acc = prec = rec = None
    for line in result.stdout.splitlines():
        ll = line.lower()
        try:
            if "f1"        in ll: f1   = float(line.strip().split()[-1])
            if "accuracy"  in ll: acc  = float(line.strip().split()[-1])
            if "precision" in ll: prec = float(line.strip().split()[-1])
            if "recall"    in ll: rec  = float(line.strip().split()[-1])
        except (ValueError, IndexError):
            pass

    if f1 is None:
        print(f"  ✗ WARNING: could not parse metrics for {variant}. Check stderr above.")

    eval_metrics[variant] = {
        "f1": f1, "accuracy": acc, "precision": prec, "recall": rec,
        "expected_f1": EXPECTED_F1[variant],
    }

# Save results
out_path = f"{RESULTS_DIR}/perception_ablation.json"
with open(out_path, "w") as f:
    json.dump(eval_metrics, f, indent=2)
print(f"\n✓ Results saved to {out_path}")

# Sanity check
print()
print(f"  {chr(39)+'Variant':<24} {'F1':>6}  {'Acc':>6}  {'Prec':>6}  {'Rec':>6}")
print("  " + "-"*52)
for v in VARIANTS:
    m = eval_metrics[v]
    f = m["f1"] or 0; a = m["accuracy"] or 0; p = m["precision"] or 0; r = m["recall"] or 0
    print(f"  {v:<24} {f:.4f}  {a:.4f}  {p:.4f}  {r:.4f}")

## 4 · Plot Table 1 Results

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

VARIANT_LABELS = {
    "radar_cnn":      "Radar CNN\n(baseline)",
    "multimodal_cnn": "Multimodal CNN\n(fusion)",
    "vit_single":     "ViT Single\n(radar only)",
    "vit_multimodal": "ViT Multimodal\n(proposed)",
}

variants  = list(VARIANTS)
ours_f1   = [eval_metrics[v].get("f1")        or 0.0 for v in variants]
ours_acc  = [eval_metrics[v].get("accuracy")  or 0.0 for v in variants]
ours_prec = [eval_metrics[v].get("precision") or 0.0 for v in variants]
ours_rec  = [eval_metrics[v].get("recall")    or 0.0 for v in variants]

x = np.arange(len(variants))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Table 1 — Perception Encoder Ablation", fontsize=13, fontweight="bold")

# Colour scheme: baseline grey, others blue/green, proposed red
COLORS = ["#94A3B8", "#60A5FA", "#34D399", "#DC2626"]

# ── Panel 1: F1 per variant (single bar, baseline vs proposed) ────────────────
ax1 = axes[0]
bars = ax1.bar(x, ours_f1, color=COLORS, alpha=0.90, edgecolor="white", linewidth=0.8)

# Baseline reference line (Radar CNN F1)
ax1.axhline(ours_f1[0], ls="--", color="#64748B", lw=1.2, label=f"Baseline (Radar CNN) F1={ours_f1[0]:.2f}")

# Value labels on top of each bar
for bar, val in zip(bars, ours_f1):
    ax1.text(bar.get_x() + bar.get_width()/2, val + 0.008,
             f"{val:.2f}", ha="center", va="bottom", fontsize=9, fontweight="bold")

# Annotate proposed variant
ax1.annotate("★ Proposed",
             xy=(x[-1], ours_f1[-1] + 0.008),
             xytext=(x[-1] - 0.6, ours_f1[-1] + 0.055),
             fontsize=8, color="#DC2626",
             arrowprops=dict(arrowstyle="->", color="#DC2626", lw=1.2))

ax1.set_xticks(x)
ax1.set_xticklabels([VARIANT_LABELS[v] for v in variants], fontsize=8.5)
ax1.set_ylabel("Macro F1")
ax1.set_title("Storm Classification F1")
ax1.set_ylim(0, 1.05)
ax1.legend(fontsize=8, loc="lower right")
ax1.grid(axis="y", alpha=0.3)

# ── Panel 2: All metrics per variant ─────────────────────────────────────────
ax2 = axes[1]
metrics_names = ["Accuracy", "Precision", "Recall", "F1"]
ours_vals = [
    [ours_acc[i], ours_prec[i], ours_rec[i], ours_f1[i]]
    for i in range(len(variants))
]
bar_w = 0.18
xs = np.arange(len(metrics_names))
for idx, (v, vals) in enumerate(zip(variants, ours_vals)):
    ax2.bar(xs + idx * bar_w, vals, bar_w,
            label=VARIANT_LABELS[v].replace("\n", " "),
            color=COLORS[idx], alpha=0.88, edgecolor="white", linewidth=0.6)

ax2.set_xticks(xs + bar_w * (len(variants) - 1) / 2)
ax2.set_xticklabels(metrics_names, fontsize=9)
ax2.set_ylabel("Score")
ax2.set_title("Evaluation Metrics by Variant")
ax2.set_ylim(0.60, 1.0)   # tighter y-axis to show differences clearly
ax2.legend(fontsize=7, loc="lower right")
ax2.grid(axis="y", alpha=0.3)

# Baseline reference line on panel 2 as well
for metric_idx, baseline_val in enumerate([ours_acc[0], ours_prec[0], ours_rec[0], ours_f1[0]]):
    ax2.plot([metric_idx - 0.15, metric_idx + 4*bar_w + 0.05],
             [baseline_val, baseline_val],
             ls="--", color="#94A3B8", lw=0.8, alpha=0.7)

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/table1_perception_ablation.pdf", bbox_inches="tight", dpi=300)
plt.show()
print("Figure saved to results/example_results/table1_perception_ablation.pdf")
